# Importing Neccessary Library

In [ ]:
import os, gc, random, optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, mean_squared_error, make_scorer
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import cross_val_score, KFold

import xgboost as xgb
import lightgbm as lgb

print("Libraries loaded successfully!")

SEED = 42
def seed_everything(seed=SEED):
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything()
print(f"Random seed set to {SEED}")

# Data Loading

In [ ]:
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv', index_col = 'Id')
X_test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv', index_col = 'Id')

print('test shape:', train.shape)
train.head()

In [ ]:
train.duplicated().sum()

In [ ]:
train.info()

# Exploratory Data Analysis

In [ ]:
plt.figure(figsize = (15, 6))
plt.title('Distribution of SalePrice')
plt.ylabel('Count')
plt.xlabel('Prices')
sns.histplot(train.SalePrice, bins = 50, kde = True)
plt.show()

In [ ]:
train.describe().T

In [ ]:
train.describe(include = 'object').T

In [ ]:
train[[a for a in train if train.isna().sum().loc[a] != 0]].isna().sum()

## Correlation

In [ ]:
corr = train.corr(numeric_only=True)

plt.figure(figsize=(25, 18))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title("Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.show()

## Categorical Visualization

In [ ]:
categorical_cols = train.select_dtypes(include = 'object').columns.to_list()

fig, axes = plt.subplots(len(categorical_cols)//2+1, 2, figsize=(20, 70))

for ax, col in zip(axes.flatten(), categorical_cols):
    counts = train[col].value_counts()
    ax.bar(counts.index.astype(str), counts.values)
    ax.set_title(col)
    
plt.tight_layout()
plt.show()

======================= MISSING VALUE HALNDING LATER ========================

# Modelling

In [ ]:
nominal_columns = [
    "MSZoning", "LotShape", "LandContour", "LotConfig", "Neighborhood", 
    "Condition1", "BldgType", "HouseStyle", "RoofStyle", "RoofMatl",     
    "Exterior1st", "Exterior2nd", "Foundation", "Heating", "CentralAir",   
    "Electrical", "GarageType", "SaleType", "SaleCondition" 
]

ordinal_columns = {
    "ExterQual":    ["Po", "Fa", "TA", "Gd", "Ex"],                     
    "ExterCond":    ["Po", "Fa", "TA", "Gd", "Ex"],                     
    "BsmtQual":     ["NA", "Po", "Fa", "TA", "Gd", "Ex"],               
    "BsmtCond":     ["NA", "Po", "Fa", "TA", "Gd", "Ex"],               
    "BsmtExposure": ["NA", "No", "Mn", "Av", "Gd"],                     
    "BsmtFinType1": ["NA", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],   
    "BsmtFinType2": ["NA", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],   
    "HeatingQC":    ["Po", "Fa", "TA", "Gd", "Ex"],                     
    "KitchenQual":  ["Po", "Fa", "TA", "Gd", "Ex"],                     
    "GarageQual":   ["NA", "Po", "Fa", "TA", "Gd", "Ex"],               
    "GarageCond":   ["NA", "Po", "Fa", "TA", "Gd", "Ex"],               
    "GarageFinish": ["NA", "Unf", "RFn", "Fin"],                        
    "PavedDrive":   ["N", "P", "Y"],                                    
    "Functional":   ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"]  
}

drop_features = [
    'Alley', 'PoolQC', 'Fence', 'MiscFeature',  
    'MiscVal', 'LowQualFinSF', '3SsnPorch',    
    'PoolArea', 'Condition2', 'Utilities', 'Street',       
]

In [ ]:
X = train.drop(columns = ['SalePrice'])
y = train.SalePrice

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size = 0.2, train_size = 0.8, random_state = 42
)

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        X['TotalSF'] = X['TotalBsmtSF'] + X['1stFlrSF'] + X['2ndFlrSF']
        X['TotalArea'] = X['GrLivArea'] + X['GarageArea'] + X['TotalBsmtSF']
        X['HouseAge'] = X['YrSold'] - X['YearBuilt']
        X['RemodAge'] = X['YrSold'] - X['YearRemodAdd']
        X['IsRemodeled'] = (X['YearBuilt'] != X['YearRemodAdd']).astype(int)
        X['TotalBath'] = (
            X['FullBath'] + 0.5 * X['HalfBath'] +
            X['BsmtFullBath'] + 0.5 * X['BsmtHalfBath']
        )
        X['TotalRooms'] = X['TotRmsAbvGrd'] + X['KitchenAbvGr']
        X['GarageAge'] = X['YrSold'] - X['GarageYrBlt']
        X['HasGarage'] = (~X['GarageType'].isna()).astype(int)
        X['HasBsmt'] = (X['TotalBsmtSF'] > 0).astype(int)
        X['Neighborhood_Condition'] = X['Neighborhood'] + '_' + X['Condition1']

        return X.drop(columns=drop_features, errors='ignore')

fe = FeatureEngineer()
X_fe = fe.fit_transform(X_train.copy())

numerical_cols = [
    col for col in X_fe.select_dtypes(exclude='object').columns
    if col not in drop_features + ['SalePrice']
]

ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=list(ordinal_columns.values())))
])

nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', RobustScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('ord', ordinal_transformer, list(ordinal_columns.keys())),
        ('nom', nominal_transformer, nominal_columns)
    ],
    remainder='drop'
)

full_pipeline = Pipeline(steps=[
    ('feature_eng', FeatureEngineer()),
    ('preprocess', preprocessor)
])

xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.030,
    max_depth=4,
    subsample=0.81,
    colsample_bytree=0.80,
    random_state=42
)

reg = Pipeline(steps=[
    ('preprocessor', full_pipeline),
    ('model', xgb_model)
])

reg.fit(X_train, y_train)
y_preds = reg.predict(X_valid)

print('RMSE :', mean_squared_error(np.log(y_preds), np.log(y_valid), squared = False))

In [ ]:
print('Predicting data X_test')

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
test_preds = np.zeros((len(X_test),))

for train_idx, val_idx in kfold.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    reg.fit(X_tr, y_tr)
    test_preds += reg.predict(X_test) / kfold.get_n_splits()
    
    preds = reg.predict(X_valid)
    print('RMSE =', np.sqrt(mean_squared_error(np.log(preds), np.log(y_valid))))
    
output = pd.DataFrame({
    'Id': X_test.index,
    'SalePrice': test_preds
})

output.to_csv('submission.csv', index=False)
print("Submission file saved as submission.csv")

In [ ]:
sns.histplot(output.SalePrice, bins = 50)